In [80]:
import sqlite3
import pandas as pd

In [81]:
# Kết nối database
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Tạo bảng course
cursor.execute('''
CREATE TABLE course (
    id INTEGER PRIMARY KEY,
    course_name TEXT
)
''')

# Tạo bảng student
cursor.execute('''
CREATE TABLE student (
    student_id INTEGER PRIMARY KEY,
    name TEXT,
    class TEXT,
    course_id INTEGER,
    score REAL
)
''')

# Thêm dữ liệu vào bảng course
courses = [
    (12, 'Giai tich'),
    (34, 'Thong ke'),
    (26, 'Tin hoc')
]
cursor.executemany('INSERT INTO course VALUES (?, ?)', courses)

# Thêm dữ liệu vào bảng student
students = [
    (1, 'Nguyen Minh Hoang', 'May Tinh', 12, 6.7),
    (2, 'Tran Thi Lan', 'Kinh Te', 34, 9.2),
    (3, 'Pham Van Nam', 'Toan Tin', None, 7.9),
    (4, 'Le Thanh Huyen', 'Toan Tin', 20, 7.2),
    (5, 'Vu Quoc Anh', 'May Tinh', 24, 8.0),
    (6, 'Dang Thuy Linh', 'May Tinh', 24, 5.5),
    (7, 'Bui Tien Dung', 'Kinh Te', 34, 9.2),
    (8, 'Ho Ngoc Mai', 'Toan Tin', 20, 8.8),
    (9, 'Duong Huu Phuc', 'Kinh Te', None, 7.2),
    (10, 'Cao Thi Hanh', 'May Tinh', None, 7.0)
]
cursor.executemany('INSERT INTO student VALUES (?, ?, ?, ?, ?)', students)
conn.commit()

Cau1

In [82]:
# a. Tích Descartes (Cross Join)
df_cartesian = pd.read_sql_query("SELECT * FROM student CROSS JOIN course", conn)

# b. JOIN các loại
inner_join = pd.read_sql_query("SELECT * FROM student INNER JOIN course ON student.course_id = course.id", conn)
left_join = pd.read_sql_query("SELECT * FROM student LEFT JOIN course ON student.course_id = course.id", conn)

# RIGHT JOIN
right_join = pd.read_sql_query("SELECT * FROM student RIGHT JOIN course ON student.course_id = course.id", conn)

# FULL OUTER JOIN
full_join = pd.read_sql_query('''
    SELECT * 
    FROM student 
    FULL OUTER JOIN course 
    ON student.course_id = course.id
''', conn)

print("INNER JOIN:")
display(inner_join)

print("LEFT JOIN:")
display(left_join)

print("RIGHT JOIN:")
display(right_join)

print("FULL OUTER JOIN:")
display(full_join)

INNER JOIN:


,student_id,name,class,course_id,score,id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,12,Giai tich
1,2,Tran Thi Lan,Kinh Te,34,9.2,34,Thong ke
2,7,Bui Tien Dung,Kinh Te,34,9.2,34,Thong ke


LEFT JOIN:


,student_id,name,class,course_id,score,id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,12.0,Giai tich
1,2,Tran Thi Lan,Kinh Te,34.0,9.2,34.0,Thong ke
2,3,Pham Van Nam,Toan Tin,NaN,7.9,NaN,None
3,4,Le Thanh Huyen,Toan Tin,20.0,7.2,NaN,None
4,5,Vu Quoc Anh,May Tinh,24.0,8.0,NaN,None
5,6,Dang Thuy Linh,May Tinh,24.0,5.5,NaN,None
6,7,Bui Tien Dung,Kinh Te,34.0,9.2,34.0,Thong ke
7,8,Ho Ngoc Mai,Toan Tin,20.0,8.8,NaN,None
8,9,Duong Huu Phuc,Kinh Te,NaN,7.2,NaN,None
9,10,Cao Thi Hanh,May Tinh,NaN,7.0,NaN,None


RIGHT JOIN:


,student_id,name,class,course_id,score,id,course_name
0,1.0,Nguyen Minh Hoang,May Tinh,12.0,6.7,12,Giai tich
1,2.0,Tran Thi Lan,Kinh Te,34.0,9.2,34,Thong ke
2,7.0,Bui Tien Dung,Kinh Te,34.0,9.2,34,Thong ke
3,NaN,None,None,NaN,NaN,26,Tin hoc


FULL OUTER JOIN:


,student_id,name,class,course_id,score,id,course_name
0,1.0,Nguyen Minh Hoang,May Tinh,12.0,6.7,12.0,Giai tich
1,2.0,Tran Thi Lan,Kinh Te,34.0,9.2,34.0,Thong ke
2,3.0,Pham Van Nam,Toan Tin,NaN,7.9,NaN,None
3,4.0,Le Thanh Huyen,Toan Tin,20.0,7.2,NaN,None
4,5.0,Vu Quoc Anh,May Tinh,24.0,8.0,NaN,None
5,6.0,Dang Thuy Linh,May Tinh,24.0,5.5,NaN,None
6,7.0,Bui Tien Dung,Kinh Te,34.0,9.2,34.0,Thong ke
7,8.0,Ho Ngoc Mai,Toan Tin,20.0,8.8,NaN,None
8,9.0,Duong Huu Phuc,Kinh Te,NaN,7.2,NaN,None
9,10.0,Cao Thi Hanh,May Tinh,NaN,7.0,NaN,None


Cau2

In [83]:
# 2.1 Cập nhật course_id còn thiếu
cursor.execute("UPDATE student SET course_id = (SELECT id FROM course ORDER BY RANDOM() LIMIT 1) WHERE course_id IS NULL")

# 2.2 Loại bỏ bản ghi có môn học không tồn tại trong bảng course (ID 20, 24 không có trong bảng course)
cursor.execute("DELETE FROM student WHERE course_id NOT IN (SELECT id FROM course)")
conn.commit()

# 2.3 Thống kê
# a. Tổng số sinh viên và điểm TB từng lớp
print("Thống kê theo lớp:")
stats_class = pd.read_sql_query('''
    SELECT class, COUNT(student_id) as total_students, AVG(score) as avg_score 
    FROM student GROUP BY class
''', conn)

display(stats_class)

# b. Tổng số sinh viên và điểm TB từng môn học
print("Thống kê theo môn học:")
stats_course = pd.read_sql_query('''
    SELECT c.course_name, COUNT(s.student_id) as total_students, AVG(s.score) as avg_score
    FROM student s JOIN course c ON s.course_id = c.id
    GROUP BY c.course_name
''', conn)

display(stats_course)

# c. Phân loại thi đua
print("Phân loại thi đua:")
classification = pd.read_sql_query('''
    SELECT c.course_name,
       AVG(s.score) as avg_score,
       CASE 
           WHEN AVG(s.score) >= 9.0 THEN 'Xuất sắc'
           WHEN AVG(s.score) >= 6.0 THEN 'Tốt'
           ELSE 'Kém'
       END as classification
FROM student s 
JOIN course c ON s.course_id = c.id
GROUP BY c.course_name
''', conn)

display(classification)

Thống kê theo lớp:


,class,total_students,avg_score
0,Kinh Te,3,8.533333
1,May Tinh,2,6.850000
2,Toan Tin,1,7.900000


Thống kê theo môn học:


,course_name,total_students,avg_score
0,Giai tich,1,6.700000
1,Thong ke,2,9.200000
2,Tin hoc,3,7.366667


Phân loại thi đua:


,course_name,avg_score,classification
0,Giai tich,6.700000,Tốt
1,Thong ke,9.200000,Xuất sắc
2,Tin hoc,7.366667,Tốt


Cau3

In [84]:
# Hàm hỗ trợ lấy Top 3 cao nhất và thấp nhất
def get_top_bottom(query_name, sql_query):
    df = pd.read_sql_query(sql_query, conn)
    print(f"Xếp hạng theo {query_name}")
    top_3 = df.nsmallest(3, 'rank_col') # Rank 1, 2, 3
    bottom_3 = df.nlargest(3, 'rank_col') # Rank lớn nhất
    
    print("Top 3:")
    display(top_3)
    
    print("Bottom 3:")
    display(bottom_3)
    
    display(df)

# a. Theo điểm số
sql_a = "SELECT *, RANK() OVER(ORDER BY score DESC) as rank_col FROM student"
# b. Theo lớp
sql_b = "SELECT *, RANK() OVER(PARTITION BY class ORDER BY score DESC) as rank_col FROM student"
# c. Theo mã môn học
sql_c = "SELECT *, RANK() OVER(PARTITION BY course_id ORDER BY score DESC) as rank_col FROM student"

get_top_bottom("Điểm số", sql_a)
get_top_bottom("Theo lớp", sql_b)
get_top_bottom("Theo môn học", sql_c)

Xếp hạng theo Điểm số
Top 3:


,student_id,name,class,course_id,score,rank_col
0,2,Tran Thi Lan,Kinh Te,34,9.2,1
1,7,Bui Tien Dung,Kinh Te,34,9.2,1
2,3,Pham Van Nam,Toan Tin,26,7.9,3


Bottom 3:


,student_id,name,class,course_id,score,rank_col
5,1,Nguyen Minh Hoang,May Tinh,12,6.7,6
4,10,Cao Thi Hanh,May Tinh,26,7.0,5
3,9,Duong Huu Phuc,Kinh Te,26,7.2,4


,student_id,name,class,course_id,score,rank_col
0,2,Tran Thi Lan,Kinh Te,34,9.2,1
1,7,Bui Tien Dung,Kinh Te,34,9.2,1
2,3,Pham Van Nam,Toan Tin,26,7.9,3
3,9,Duong Huu Phuc,Kinh Te,26,7.2,4
4,10,Cao Thi Hanh,May Tinh,26,7.0,5
5,1,Nguyen Minh Hoang,May Tinh,12,6.7,6


...

Xếp hạng theo Theo môn học
Top 3:


,student_id,name,class,course_id,score,rank_col
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,1
1,3,Pham Van Nam,Toan Tin,26,7.9,1
4,2,Tran Thi Lan,Kinh Te,34,9.2,1


Bottom 3:


,student_id,name,class,course_id,score,rank_col
3,10,Cao Thi Hanh,May Tinh,26,7.0,3
2,9,Duong Huu Phuc,Kinh Te,26,7.2,2
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,1


,student_id,name,class,course_id,score,rank_col
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,1
1,3,Pham Van Nam,Toan Tin,26,7.9,1
2,9,Duong Huu Phuc,Kinh Te,26,7.2,2
3,10,Cao Thi Hanh,May Tinh,26,7.0,3
4,2,Tran Thi Lan,Kinh Te,34,9.2,1
5,7,Bui Tien Dung,Kinh Te,34,9.2,1


Cau4

In [85]:
# Thêm cột mới
try:
    cursor.execute("ALTER TABLE student ADD COLUMN graduation_date DATETIME")
except:
    pass # Nếu đã tồn tại cột thì bỏ qua

# Cập nhật ngày tốt nghiệp dựa trên thứ hạng điểm số
# Rank càng cao (số 1) thì tốt nghiệp càng sớm
cursor.execute('''
    WITH RankedStudents AS (
        SELECT student_id, RANK() OVER(ORDER BY score DESC) as rnk
        FROM student
    )
    UPDATE student
    SET graduation_date = datetime('now', '+' || (SELECT rnk FROM RankedStudents WHERE RankedStudents.student_id = student.student_id) || ' days')
''')
conn.commit()

# Hiển thị kết quả cuối cùng
final_result = pd.read_sql_query("SELECT * FROM student", conn)
display(final_result)

,student_id,name,class,course_id,score,graduation_date
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,2026-04-07 16:35:24
1,2,Tran Thi Lan,Kinh Te,34,9.2,2026-04-02 16:35:24
2,3,Pham Van Nam,Toan Tin,26,7.9,2026-04-04 16:35:24
3,7,Bui Tien Dung,Kinh Te,34,9.2,2026-04-02 16:35:24
4,9,Duong Huu Phuc,Kinh Te,26,7.2,2026-04-05 16:35:24
5,10,Cao Thi Hanh,May Tinh,26,7.0,2026-04-06 16:35:24
